In [2]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow
import unicodedata



# Carregando variáveis de ambiente
load_dotenv()

True

In [3]:
# Variáveis de ambiente para autenticação e configuração do BigQuery
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("PROJECT_ID")

# Tabela SILVER no BigQuery
tabela_silver = os.getenv("SILVER_TABLE")

# Tabela GOLD no BigQuery
# Dim Calendario
table_gold_calendario = os.getenv("GOLD_CALENDARIO")

In [4]:
# Inicializa o cliente do BigQuery usando credenciais de serviço
# O parâmetro 'project_id' especifica o projeto GCP onde as queries serão executadas
client = bigquery.Client.from_service_account_json(credencial, project=project_id)


query = f"""
SELECT *
FROM `{tabela_silver}`
"""

In [5]:
# Executa e converte pra DataFrame
resultado = client.query(query)
df = resultado.to_dataframe()

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [6]:
# Criando DataFrame de datas únicas e renomeando a coluna para "data"
df_date = df[["data_da_coleta"]].drop_duplicates().reset_index(drop=True).rename(columns={"data_da_coleta": "data"})

In [ ]:
#  Criando Dataframe do calendário a partir das datas únicas
df_calendario = df_date

In [8]:
# Criando um dicionario para mapear os meses em português
meses_pt = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}


In [ ]:
# Convertendo a coluna "data" para datetime e extraindo as informações necessárias para criar as colunas do calendário
datas = pd.to_datetime(df_calendario["data"])

df_calendario["data_sk"]       = datas.dt.strftime("%Y%m%d").astype(int)
df_calendario["dia"]           = datas.dt.day
df_calendario["mes"]           = datas.dt.month
df_calendario["mes_nome"]      = datas.dt.month.map(meses_pt)
df_calendario["ano"]           = datas.dt.year
df_calendario["mes_ano"]       = datas.dt.strftime("%m/%Y")
df_calendario["trimestre"]     = datas.dt.quarter
df_calendario["semestre"]      = datas.dt.month.apply(lambda x: 1 if x <= 6 else 2)
df_calendario["semana_do_ano"] = datas.dt.isocalendar().week.astype(int)

In [10]:
# Reordena colocando data_sk como primeira coluna
cols = ["data_sk"] + [c for c in df_calendario.columns if c != "data_sk"]
df_calendario = df_calendario[cols]

In [12]:
# Criação e carga da tabela Silver

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Silver seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_calendario,
    table_gold_calendario,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Gold_Calendario criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Gold_Calendario criada e dados carregados com sucesso
